In [14]:
import pandas as pd
from urllib.parse import urlparse
import re
import math
import numpy as np
import zlib



In [2]:
df = pd.read_csv('C:/Users/USER/Desktop/URL-Phishing-Detection/datasets/raw_data/StealthPhisher2025.csv')

In [3]:
df.head(5)

,URL,LengthOfURL,Domain,URLComplexity,CharacterComplexity,DomainLengthOfURL,IsDomainIP,TLD,TLDLength,LetterCntInURL,...,UniqueFeatureCnt,WAPLegitimate,WAPPhishing,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt,LikelinessIndex,Label
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,84,bafkreibre4pwizu3d73y7at37ewy6nhklfhb4mb75tp25...,14.000,0.013158,76,0,com,3,55,...,0,0.036909,0.034343,4.739567,1.00,1.000000,1,3,0.263200,Phishing
1,http://101.200.220.118:8090/ledshow2.exe,40,101.200.220.118:8090,12.000,0.030303,20,1,118:8090,8,10,...,0,0.030220,0.031712,3.808271,1.00,1.222222,5,3,0.329167,Phishing
2,https://1llc5nv.duckdns.org/,28,1llc5nv.duckdns.org,9.000,0.050000,19,0,org,3,15,...,0,0.043893,0.038140,4.056021,0.75,1.307692,0,2,0.684211,Phishing
3,http://hrga.melonwoodhomes.com/,31,hrga.melonwoodhomes.com,9.375,0.041667,23,0,com,3,21,...,0,0.061778,0.046342,3.827833,0.75,1.275862,0,2,0.608696,Phishing
4,https://www.aspb.gob.bo,23,www.aspb.gob.bo,5.875,0.066667,15,0,bo,2,9,...,10,0.073055,0.046233,3.346439,1.00,1.400000,0,2,0.733333,Legitimate


In [4]:
df.shape

(336749, 65)

In [5]:
df= df[["URL", "Label"]]

In [6]:
df.head()

,URL,Label
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,Phishing
1,http://101.200.220.118:8090/ledshow2.exe,Phishing
2,https://1llc5nv.duckdns.org/,Phishing
3,http://hrga.melonwoodhomes.com/,Phishing
4,https://www.aspb.gob.bo,Legitimate


In [7]:
# Label sütununu 1 ve 0 haline getiriyoruz.
df["Label"] = df["Label"].map({
    "Legitimate": 0,
    "Phishing": 1
})

In [11]:
df.head()

,URL,Label,LengthOfURL
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,1,84
1,http://101.200.220.118:8090/ledshow2.exe,1,40
2,https://1llc5nv.duckdns.org/,1,28
3,http://hrga.melonwoodhomes.com/,1,31
4,https://www.aspb.gob.bo,0,23


In [10]:
df["LengthOfURL"] = df["URL"].astype(str).apply(len)

In [15]:
def check_bad_urls(x):
    try:
        len(urlparse(x).netloc)
        return False
    except ValueError:
        return True

df[df["URL"].apply(check_bad_urls)]

,URL,Label,LengthOfURL


In [17]:
# [.] ifadesini . olarak değiştirdik.
df["URL"] = df["URL"].astype(str).str.replace(r"\[\.\]", ".", regex=True)

In [18]:
df[df["URL"].apply(check_bad_urls)]

,URL,Label,LengthOfURL


In [20]:
df["DomainLengthOfURL"] = df["URL"].apply(
    lambda x: len(urlparse(x).netloc)
)

In [21]:
df["TLDLength"] = df["URL"].astype(str).apply(
    lambda x: (
        lambda parsed: len(parsed.netloc.split(":")[0].split(".")[-1])
        if "." in parsed.netloc else 0
    )(urlparse(x if "://" in x else "http://" + x))
)

In [22]:
df["IsDomainIP"] = df["URL"].astype(str).apply(
    lambda x: 1 if re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", urlparse(x).netloc.split(":")[0]) else 0
)

In [23]:
# URL içinde kaç tane rakam
df["DigitCntInURL"] = df["URL"].astype(str).apply(
    lambda x: sum(c.isdigit() for c in x)
)

In [24]:
df["EqualCharCntInURL"] = df["URL"].astype(str).apply(
    lambda x: x.count("=")
)

In [25]:
df["QuesMarkCntInURL"] = df["URL"].astype(str).apply(
    lambda x: x.count("?")
)

In [26]:
df["OtherSpclCharCntInURL"] = df["URL"].astype(str).apply(
    lambda x: len(re.findall(r"[^\w]", x))
)

In [27]:
df["URLLetterRatio"] = df["URL"].astype(str).apply(
    lambda x: (sum(c.isalpha() for c in x) / len(x)) if len(x) > 0 else 0
)

In [28]:
df["HavingPath"] = df["URL"].astype(str).apply(
    lambda x: 1 if urlparse(x if "://" in x else "http://" + x).path not in ["", "/"] else 0
)

In [29]:
df["HavingAnchor"] = df["URL"].astype(str).apply(
    lambda x: 1 if "#" in x else 0
)

In [30]:
df["PathLength"] = df["URL"].astype(str).apply(
    lambda x: len(urlparse(x if "://" in x else "http://" + x).path)
)

In [32]:
def shannon_entropy(s):
    s = str(s)
    if len(s) == 0:
        return 0
    prob = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in prob)

df["ShannonEntropy"] = df["URL"].astype(str).apply(shannon_entropy)

In [33]:
df["FractalDimension"] = df.apply(
    lambda row: (
        row["ShannonEntropy"] / np.log(row["LengthOfURL"])
        if row["LengthOfURL"] > 1 else 0
    ),
    axis=1
)

In [34]:
df["KolmogorovComplexity"] = df["URL"].astype(str).apply(
    lambda x: (
        len(zlib.compress(x.encode())) / len(x)
        if len(x) > 0 else 0
    )
)

In [35]:
df["HexPatternCnt"] = df["URL"].astype(str).apply(
    lambda x: len(re.findall(r"%[0-9A-Fa-f]{2}", x))
)

In [36]:
df["Base64PatternCnt"] = df["URL"].astype(str).apply(
    lambda x: len(re.findall(r"[A-Za-z0-9+/]{20,}={0,2}", x))
)

In [37]:
df.shape

(336749, 19)

In [38]:
df.head()

,URL,Label,LengthOfURL,DomainLengthOfURL,TLDLength,IsDomainIP,DigitCntInURL,EqualCharCntInURL,QuesMarkCntInURL,OtherSpclCharCntInURL,URLLetterRatio,HavingPath,HavingAnchor,PathLength,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,1,84,76,3,0,17,0,0,7,0.714286,0,0,0,4.792582,1.081647,0.988095,0,1
1,http://101.200.220.118:8090/ledshow2.exe,1,40,20,3,1,17,0,0,9,0.350000,1,0,13,3.896439,1.056266,1.200000,0,0
2,https://1llc5nv.duckdns.org/,1,28,19,3,0,2,0,0,6,0.714286,0,0,1,4.137538,1.241682,1.285714,0,0
3,http://hrga.melonwoodhomes.com/,1,31,23,3,0,0,0,0,6,0.806452,0,0,1,3.925993,1.143275,1.258065,0,0
4,https://www.aspb.gob.bo,0,23,15,2,0,0,0,0,6,0.739130,0,0,0,3.468577,1.106230,1.347826,0,0


In [39]:
df.tail()

,URL,Label,LengthOfURL,DomainLengthOfURL,TLDLength,IsDomainIP,DigitCntInURL,EqualCharCntInURL,QuesMarkCntInURL,OtherSpclCharCntInURL,URLLetterRatio,HavingPath,HavingAnchor,PathLength,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt
336744,https://www.theboothcompany.ca,0,30,22,2,0,0,0,0,5,0.833333,0,0,0,3.831402,1.126486,1.266667,0,0
336745,https://www.biscaynetimes.com,0,29,21,3,0,0,0,0,5,0.827586,0,0,0,3.952303,1.173732,1.275862,0,0
336746,https://sbvkt5.firebaseapp.com/,1,31,22,3,0,1,0,0,6,0.774194,0,0,1,4.082598,1.188880,1.258065,0,0
336747,http://117.219.119.58:47440/Mozi.a,1,34,20,2,1,16,0,0,9,0.264706,1,0,7,3.995715,1.133099,1.176471,0,0
336748,https://www.cottonwoodmeadow.com,0,32,24,3,0,0,0,0,5,0.843750,0,0,0,3.590018,1.035860,1.187500,0,0
